# 🚀 Nexa 3D Gen — Servidor GPU no Colab

Este notebook roda o **Hunyuan3D-2** em uma GPU T4 (16GB VRAM) do Google Colab.

Ele cria um servidor API exposto via **ngrok**, que o seu frontend local se conecta para gerar modelos 3D em minutos.

---

**Instruções:**
1. Vá em `Ambiente de execução > Alterar tipo de ambiente de execução` e selecione **T4 GPU**
2. Clique em `Ambiente de execução > Executar tudo`
3. Copie a URL do ngrok que aparecerá na última célula
4. Cole na barra de conexão do seu frontend local e pressione Enter

In [ ]:
# Célula 1: Verificar GPU
!nvidia-smi
import torch
print(f'\n✅ PyTorch {torch.__version__}')
print(f'✅ CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Célula 2: Instalar dependências
!pip install -q fastapi uvicorn pyngrok python-multipart trimesh pymeshlab
!pip install -q diffusers einops opencv-python-headless

# Clonar Hunyuan3D-2
!git clone https://github.com/Tencent/Hunyuan3D-2.git 2>/dev/null || echo 'Repositório já clonado'
!cd Hunyuan3D-2 && pip install -q -r requirements.txt

print('\n✅ Dependências instaladas!')

In [ ]:
# Célula 3: Baixar e carregar modelos
import sys
sys.path.append('/content/Hunyuan3D-2')

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
from hy3dgen.texgen import Hunyuan3DPaintPipeline
from hy3dgen.rembg import BackgroundRemover
from hy3dgen.shapegen.postprocessors import FaceReducer, FloaterRemover, DegenerateFaceRemover

print('⏳ Carregando Shape Pipeline (FP16)...')
shape_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2',
    subfolder='hunyuan3d-dit-v2-0',
    torch_dtype=torch.float16
)
shape_pipeline.to('cuda')

# Paint Pipeline (normal)
paint_pipeline = None
try:
    print('⏳ Carregando Paint Pipeline (FP16)...')
    paint_pipeline = Hunyuan3DPaintPipeline.from_pretrained(
        'tencent/Hunyuan3D-2',
        subfolder='hunyuan3d-paint-v2-0'
    )
    paint_pipeline.to('cuda')
    print('✅ Paint Pipeline carregado!')
except Exception as e:
    print(f'⚠️ Paint Pipeline não disponível: {e}')

# Paint Pipeline Turbo (mais rápido)
paint_pipeline_turbo = None
try:
    print('⏳ Carregando Paint Pipeline Turbo...')
    paint_pipeline_turbo = Hunyuan3DPaintPipeline.from_pretrained(
        'tencent/Hunyuan3D-2',
        subfolder='hunyuan3d-paint-v2-0-turbo'
    )
    paint_pipeline_turbo.to('cuda')
    print('✅ Paint Pipeline Turbo carregado!')
except Exception as e:
    print(f'⚠️ Paint Pipeline Turbo não disponível: {e}')

# Background Remover
bg_remover = None
try:
    bg_remover = BackgroundRemover()
    print('✅ Background Remover carregado!')
except Exception as e:
    print(f'⚠️ Background Remover não disponível: {e}')

# Post-processors
face_reducer = FaceReducer()
floater_remover = FloaterRemover()
degenerate_remover = DegenerateFaceRemover()

print('\n✅ Todos os modelos carregados com sucesso!')

In [ ]:
# Célula 4: Iniciar servidor API + ngrok

import os
import uuid
import shutil
import asyncio
import threading
import trimesh
import numpy as np
from contextlib import asynccontextmanager
from typing import Dict
from PIL import Image

from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.staticfiles import StaticFiles
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import uvicorn
from pyngrok import ngrok

# Diretórios
os.makedirs('uploads', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Estado
tasks: Dict[str, Dict] = {}
task_queue = asyncio.Queue()

def get_mesh_stats(mesh):
    """Extrai estatísticas da malha para o frontend"""
    stats = {}
    try:
        if isinstance(mesh, trimesh.Scene):
            combined = trimesh.util.concatenate(
                [g for g in mesh.geometry.values() if isinstance(g, trimesh.Trimesh)]
            )
            mesh = combined
        stats['vertices'] = len(mesh.vertices)
        stats['faces'] = len(mesh.faces)
        stats['watertight'] = bool(mesh.is_watertight)
        bounds = mesh.bounds
        dims = bounds[1] - bounds[0]
        stats['dimensions'] = {'x': round(float(dims[0]), 3), 'y': round(float(dims[1]), 3), 'z': round(float(dims[2]), 3)}
        try:
            stats['volume'] = round(float(mesh.volume), 4) if mesh.is_watertight else None
        except:
            stats['volume'] = None
    except Exception as e:
        print(f'Erro ao calcular stats: {e}')
        stats = {'vertices': 0, 'faces': 0, 'watertight': False, 'dimensions': {'x':0,'y':0,'z':0}, 'volume': None}
    return stats

def process_sync(task_id, image_path, settings):
    try:
        tasks[task_id]['status'] = 'processing'
        steps = settings.get('steps', 30)
        guidance = settings.get('guidance_scale', 7.5)
        dual_guidance = settings.get('dual_guidance_scale', 10.5)
        octree_res = settings.get('octree_resolution', 384)
        max_faces = settings.get('max_faces', 80000)
        export_fmt = settings.get('export_format', 'glb')
        seed = settings.get('seed', None)
        do_remove_bg = settings.get('remove_bg', True)
        do_turbo = settings.get('turbo_texture', False)
        do_remove_floaters = settings.get('remove_floaters', True)
        do_remove_degenerate = settings.get('remove_degenerate_faces', True)

        # Pré-processamento: remover fundo
        img = Image.open(image_path).convert('RGBA')
        if do_remove_bg and bg_remover:
            print(f'[{task_id[:8]}] Removendo fundo...')
            try:
                img = bg_remover(img)
            except Exception as e:
                print(f'[{task_id[:8]}] Remoção de fundo falhou: {e}')

        generator = None
        if seed is not None:
            generator = torch.Generator(device='cuda').manual_seed(int(seed))

        print(f'[{task_id[:8]}] Gerando forma... steps={steps}, octree={octree_res}, guidance={guidance}')
        mesh = shape_pipeline(
            image=img,
            num_inference_steps=steps,
            guidance_scale=guidance,
            dual_guidance_scale=dual_guidance,
            octree_resolution=octree_res,
            generator=generator
        )[0]

        # Texturização
        active_paint = paint_pipeline_turbo if (do_turbo and paint_pipeline_turbo) else paint_pipeline
        if active_paint:
            print(f'[{task_id[:8]}] Texturizando{" (turbo)" if do_turbo else ""}...')
            try:
                mesh = active_paint(mesh, image=img)
            except Exception as e:
                print(f'[{task_id[:8]}] Textura ignorada: {e}')

        # Pós-processamento
        if do_remove_floaters:
            print(f'[{task_id[:8]}] Removendo flutuantes...')
            try:
                mesh = floater_remover(mesh)
            except Exception as e:
                print(f'[{task_id[:8]}] FloaterRemover falhou: {e}')

        if do_remove_degenerate:
            print(f'[{task_id[:8]}] Limpando faces degeneradas...')
            try:
                mesh = degenerate_remover(mesh)
            except Exception as e:
                print(f'[{task_id[:8]}] DegenerateRemover falhou: {e}')

        # Reduzir faces
        if max_faces < 200000:
            print(f'[{task_id[:8]}] Reduzindo para {max_faces} faces...')
            try:
                mesh = face_reducer(mesh, max_facenum=max_faces)
            except Exception as e:
                print(f'[{task_id[:8]}] FaceReducer falhou: {e}')

        # Estatísticas
        stats = get_mesh_stats(mesh)
        tasks[task_id]['mesh_stats'] = stats
        print(f'[{task_id[:8]}] Stats: {stats["vertices"]} vértices, {stats["faces"]} faces, watertight={stats["watertight"]}')

        # Exportar
        task_folder = os.path.join('models', task_id)
        os.makedirs(task_folder, exist_ok=True)

        # Exportar formato principal
        filename = f'{task_id}.{export_fmt}'
        output_path = os.path.join(task_folder, filename)
        mesh.export(output_path, file_type=export_fmt)

        # Exportar formatos adicionais
        export_urls = {}
        for fmt in ['glb', 'stl', 'obj']:
            fmt_filename = f'{task_id}.{fmt}'
            fmt_path = os.path.join(task_folder, fmt_filename)
            if not os.path.exists(fmt_path):
                try:
                    mesh.export(fmt_path, file_type=fmt)
                except Exception as e:
                    print(f'[{task_id[:8]}] Export {fmt} falhou: {e}')
            if os.path.exists(fmt_path):
                export_urls[fmt] = f'/models/{task_id}/{fmt_filename}'

        tasks[task_id]['status'] = 'completed'
        tasks[task_id]['model_url'] = f'/models/{task_id}/{filename}'
        tasks[task_id]['export_urls'] = export_urls
        print(f'[{task_id[:8]}] ✅ Concluído!')
    except Exception as e:
        print(f'[{task_id[:8]}] ❌ Falhou: {e}')
        import traceback
        traceback.print_exc()
        tasks[task_id]['status'] = 'failed'

async def worker():
    print('🔄 Worker pronto. Aguardando tarefas...')
    while True:
        task_id, image_path, settings = await task_queue.get()
        await asyncio.to_thread(process_sync, task_id, image_path, settings)
        task_queue.task_done()

@asynccontextmanager
async def lifespan(app):
    t = asyncio.create_task(worker())
    yield
    t.cancel()

app = FastAPI(lifespan=lifespan)
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

@app.post('/generate')
async def generate(
    image: UploadFile = File(...),
    steps: int = Form(30),
    seed: int = Form(-1),
    guidance_scale: float = Form(7.5),
    dual_guidance_scale: float = Form(10.5),
    octree_resolution: int = Form(384),
    max_faces: int = Form(80000),
    export_format: str = Form('glb'),
    remove_bg: bool = Form(True),
    turbo_texture: bool = Form(False),
    remove_floaters: bool = Form(True),
    remove_degenerate_faces: bool = Form(True),
):
    task_id = str(uuid.uuid4())
    ext = image.filename.split('.')[-1]
    path = f'uploads/{task_id}.{ext}'
    with open(path, 'wb') as f:
        shutil.copyfileobj(image.file, f)
    settings_dict = {
        'steps': steps, 'seed': seed if seed != -1 else None,
        'guidance_scale': guidance_scale, 'dual_guidance_scale': dual_guidance_scale,
        'octree_resolution': octree_resolution, 'max_faces': max_faces,
        'export_format': export_format, 'remove_bg': remove_bg,
        'turbo_texture': turbo_texture, 'remove_floaters': remove_floaters,
        'remove_degenerate_faces': remove_degenerate_faces,
    }
    tasks[task_id] = {
        'status': 'queued', 'model_url': None,
        'settings': settings_dict, 'mesh_stats': None, 'export_urls': {}
    }
    await task_queue.put((task_id, path, settings_dict))
    return {'task_id': task_id}

@app.get('/status/{task_id}')
async def status(task_id: str):
    if task_id not in tasks:
        raise HTTPException(404, 'Não encontrado')
    data = tasks[task_id].copy()
    if data['status'] == 'queued':
        data['queue_position'] = task_queue.qsize()
    return data

app.mount('/models', StaticFiles(directory='models'), name='models')

# --- NGROK ---
# Coloque seu token ngrok aqui:
# ngrok.set_auth_token('SEU_TOKEN_AQUI')

public_url = ngrok.connect(8000)
print(f'\n{"="*60}')
print(f'🌐 SERVIDOR PRONTO!')
print(f'🔗 URL Pública: {public_url}')
print(f'{"="*60}')
print(f'\nCopie a URL acima e cole na barra de conexão do seu frontend!')

# Rodar servidor
threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000), daemon=True).start()

In [ ]:
# Célula 5: Manter notebook ativo
import time
print('⏳ Servidor rodando... NÃO feche esta aba do Colab.')
print(f'🔗 URL: {public_url}')
while True:
    time.sleep(60)
    print(f'[{time.strftime("%H:%M")}] Servidor ativo ✅')